# vlm 赛道 · 数据集/题目分析（for 题目构造）

合并自原 `eval_analysis.ipynb` 与 `eval_review.ipynb` 的 vlm 视角。

**当前状态（占位）**：vlm 赛道暂不拆代码——抽样（`eval_sample.py`）与出题/判分协议待拍板后再落本子模块；数据区 `vlm/data/` 由用户重抽时指定落盘。本册先提供**题库审阅骨架**：默认读 `bagel/results/wkbench_v0/questions.jsonl`（runner 快照，只筛 task=vlm），换题库改 `QFILE` 即可。

In [ ]:
import sys, subprocess
import json as _json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML
plt.rcParams['font.family'] = ['Noto Sans CJK SC', 'DejaVu Sans']  # CJK 优先，避免中文豆腐块
plt.rcParams['axes.unicode_minus'] = False

## 评测集构建（TODO）

vlm 赛道抽样协议待拍板（独立打分/出题口径确定后，把 `eval_sample.py` 拆入本子模块，此处补委托调用格）。重抽时把数据落到 `vlm/data/`。

In [ ]:
# ---- 题库加载（默认读 wkbench runner 快照；换题库改 QFILE）----
QFILE = Path('../../bagel/results/wkbench_v0/questions.jsonl')
EVAL_DIR = Path('data')      # 样本图根（重抽后落 vlm/data/）
if not QFILE.exists():
    qs = []
    print(f'!! 题库不存在：{QFILE} —— 改 QFILE 指向已有题库快照')
else:
    qs = [_json.loads(l) for l in QFILE.open(encoding='utf-8')
          if _json.loads(l).get('task') == 'vlm']
    by_sample = {}
    for q in qs:
        by_sample.setdefault(q['_sample_image'], []).append(q)
    print(f'vlm {len(qs)} 题 / {len(by_sample)} 个样本')

In [ ]:
# ---- 题库打印工具（只读展示）----
from IPython.display import Image as _NBImage

W = 74

def _pr(items, key):
    for p in items or []:
        if isinstance(p, dict):
            w = p.get('weight')
            txt = p.get(key) or p.get('point') or p.get('check') or ''
            print(f"  - {('[' + str(w) + '] ') if w is not None else ''}{txt}")
        else:
            print(f"  - {p}")

def _print_question(q):
    task = (q.get('task') or '?').upper()
    print('\n' + '-' * W)
    print(f"[{task} | {q.get('difficulty', '?')} | {q.get('qid', '?')} | "
          f"知识维度: {q.get('knowledge_dim', '?')}]"
          + (f" | edit_type: {q.get('edit_type')} | suite: {q.get('suite')}" if q.get('edit_type') else ''))
    print(f"探针维度: {'、'.join(q.get('probe_dims') or [])}")
    prompt = q.get('stem') or q.get('edit_instruction') or q.get('gen_prompt') or ''
    print(f"\n[指令]\n{prompt}")
    if q.get('choices'):
        print(f"\n[选项]\n{q['choices']}")
    if q.get('reference_answer'):
        print(f"\n[参考答案]\n{q['reference_answer']}")
    if q.get('rubric'):
        print('\n[判分要点]'); _pr(q.get('rubric'), 'point')
    if q.get('expected_changes'):
        print('\n[执行到位要点]'); _pr(q.get('expected_changes'), 'point')
    if q.get('preserved_elements'):
        print('\n[保持一致要点]'); _pr(q.get('preserved_elements'), 'point')
    if q.get('implicit_checks'):
        print('\n[隐含知识校验点]'); _pr(q.get('implicit_checks'), 'check')
    print(f"\n[推理链] {q.get('reasoning_chain', '')}")
    fm = q.get('expected_failure_modes') or []
    if fm:
        print('\n[预期失败模式]')
        for m in fm:
            print(f"  * {m}")
    ea = q.get('evidence_audit') or {}
    if ea:
        print('\n[证据审计]')
        print('  visible_facts:', '；'.join(ea.get('visible_facts') or []))
        print('  answerable_by_image:', '；'.join(ea.get('answerable_by_image') or []))
    if q.get('needs_verification'):
        print('  !!! 需人工核验')

def _print_sample(img_rel, group, img_width=480):
    label = group[0].get('_query_label', '')
    sid = img_rel.split('/')[1][:4]
    img_path = EVAL_DIR / img_rel

    print('\n' + '=' * W)
    print(f'>>> 样本 {sid} | {label} | {len(group)} 题')
    print('=' * W)
    try:
        meta = next(_json.loads(l) for l in (EVAL_DIR / 'samples.jsonl').open(encoding='utf-8')
                    if _json.loads(l)['image'] == img_rel)
        print('caption:', meta.get('caption', ''))
    except (StopIteration, OSError):
        pass
    if img_path.exists():
        display(_NBImage(filename=str(img_path), width=img_width))
    else:
        print('!! 缺图:', img_path)
    for q in group:
        _print_question(q)

In [ ]:
# 可用取值速览：过滤参数能填什么，看这里
from collections import Counter

def _cnt(items):
    return ' | '.join(f'{k}:{v}' for k, v in sorted(Counter(items).items()))

if not qs:
    print('题库为空，跳过（先跑 eval_synthesize.py 出题）')
else:
    print('difficulty   :', _cnt(q.get('difficulty') for q in qs))
    print('knowledge_dim:', _cnt(q.get('knowledge_dim') for q in qs))
    print('probe_dims   :', _cnt(p for q in qs for p in (q.get('probe_dims') or [])))
    print('sample 标签  :', _cnt(sorted({q['_query_label'] for q in qs})))
    print('qid          :', ', '.join(q['qid'] for q in qs))

In [ ]:
# ====== 过滤 / 抽样参数（改完重跑本 cell 即可；空值 = 不过滤）======
QIDS = []              # 精确指定题目
SAMPLES = []           # 按样本筛：样本号前缀或 query 标签，如 ['0001', '厦门方特梦幻王国']
DIFFICULTIES = []      # 难度：'L1' / 'L2' / 'L3'
KNOWLEDGE_DIMS = []    # 知识维度，如 ['physics_commonsense', 'film_anime_game']
PROBE_DIMS = []        # 探针维度（题目命中任一即保留），如 ['hallucination_resist']
ONLY_NEEDS_VERIFY = False   # True = 只看自报需人工核验的题
SAMPLE_N = 0           # >0 时按样本抽：随机抽 N 个样本（与上面过滤叠加），0 = 不抽
SEED = 7               # 抽样随机种子，固定可复现；换种子换一批
IMG_WIDTH = 480        # 图片显示宽度
# ===================================================================

def _keep(q):
    if QIDS and q['qid'] not in QIDS:
        return False
    if SAMPLES:
        sid = q['_sample_image'].split('/')[1][:4]
        if not (sid in SAMPLES or q.get('_query_label') in SAMPLES):
            return False
    if DIFFICULTIES and q.get('difficulty') not in DIFFICULTIES:
        return False
    if KNOWLEDGE_DIMS and q.get('knowledge_dim') not in KNOWLEDGE_DIMS:
        return False
    if PROBE_DIMS and not (set(PROBE_DIMS) & set(q.get('probe_dims') or [])):
        return False
    if ONLY_NEEDS_VERIFY and not q.get('needs_verification'):
        return False
    return True

if not qs:
    print('题库为空，跳过（先跑 eval_synthesize.py 出题）')
else:
    _sel = [q for q in qs if _keep(q)]
    if SAMPLE_N > 0:
        import random
        pool = sorted({q['_sample_image'] for q in _sel})
        if len(pool) > SAMPLE_N:
            pool = set(random.Random(SEED).sample(pool, SAMPLE_N))
        _sel = [q for q in _sel if q['_sample_image'] in pool]

    print(f'命中 {len(_sel)} 题 / {len({q["_sample_image"] for q in _sel})} 个样本（全库 {len(qs)} 题）')
    for _img, _group in sorted({k: [q for q in _sel if q['_sample_image'] == k]
                                for k in {q['_sample_image'] for q in _sel}}.items()):
        _print_sample(_img, _group, IMG_WIDTH)